# Task 1 & 2 prompting trên Qwen3.5-4B (Kaggle GPU T4 ×2)

So sánh với fine-tune của nhóm trong **cùng điều kiện**: Qwen/Qwen3.5-4B revision `851bf6e8…`, nén 4-bit NF4 (double quant),
FP16, greedy, tắt thinking, seed 42, GPU T4. Khác nhau chỉ ở phương pháp: prompt + RAG thay cho adapter QLoRA.

| Bản | Cách làm | Input cho model |
|---|---|---|
| P0 | zero-shot | đề + code |
| P1 | RAG BM25 k=1 (1 bài mẫu, chỉ nhãn/điểm, không feedback) | đề + code |
| P2 | RAG BM25 k=1 | đề + code + kết quả biên dịch + kết quả chạy test do LLM sinh |

**Cách chạy**
1. Settings → Accelerator → **GPU T4 x2**, bật **Internet** (tải model từ Hugging Face).
2. Add Input → dataset **private** chứa `llm-apa-qwen4b.zip` (Kaggle có thể tự giải nén, không sao).
3. Chạy cell smoke trước để xem tốc độ. Sau đó dùng **Save Version → Save & Run All (Commit)** để chạy nền (tối đa 12 giờ),
   tắt trình duyệt cũng được. Xong tải `/kaggle/working/qwen4b_results.zip`.

Không cần API key: bộ test, kết quả biên dịch và chạy test đã có sẵn trong zip (không gọi Groq).

In [ ]:
from pathlib import Path
import os, sys, json, shutil, zipfile, subprocess, time

INPUT, WORK = Path('/kaggle/input'), Path('/kaggle/working')
BUNDLE = WORK / 'bundle'
if not (BUNDLE / 'LLM-APA/scripts/run_all.py').exists():
    roots = sorted({p.parents[2] for p in INPUT.rglob('LLM-APA/scripts/run_all.py')})
    zips = list(INPUT.rglob('llm-apa-qwen4b.zip'))
    if len(roots) == 1:
        shutil.copytree(roots[0], BUNDLE, dirs_exist_ok=True)
        print('Copied from', roots[0])
    elif not roots and len(zips) == 1:
        with zipfile.ZipFile(zips[0]) as z:
            z.extractall(BUNDLE)
        print('Unzipped', zips[0])
    else:
        raise ValueError(f'Need exactly one bundle. Folders: {roots}; zips: {zips}')
REPO = BUNDLE / 'LLM-APA'
TRAIN = BUNDLE / 'data/sample_dataset'
TEST = BUNDLE / 'data/public_test_release'
assert (TRAIN / 'exams.json').exists() and (TEST / 'exams.json').exists()

# Keep Kaggle's CUDA PyTorch; pin the libraries the project uses (pyproject.toml).
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==5.18.0', 'accelerate==1.15.0',
                'bitsandbytes>=0.49,<1', 'pyyaml', 'scikit-learn', 'pandas', 'tqdm'], check=True)

# Unzipping resets file times; make saved test results newer than the suites so they are reused, not rerun.
now = time.time()
for f in (REPO / 'outputs/datasets').rglob('*.json'):
    os.utime(f, (now + 60, now + 60))
os.chdir(REPO)
print('Repo:', REPO)

In [ ]:
import torch
print('PyTorch', torch.__version__, 'CUDA', torch.version.cuda)
assert torch.cuda.device_count() == 2, 'Choose GPU T4 x2 in Settings'
for i in range(2):
    p = torch.cuda.get_device_properties(i)
    print(i, p.name, round(p.total_memory / 2**30, 1), 'GiB')

from huggingface_hub import snapshot_download
MODEL, REVISION = 'Qwen/Qwen3.5-4B', '851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a'  # same as the fine-tuned model
snapshot_download(MODEL, revision=REVISION, allow_patterns=['*.json', '*.safetensors', '*.txt', '*.jinja', '*.model'])
print('Model ready:', MODEL, REVISION)

# Task 1 only by default; add 'task2' to also run Task 2 (about twice the time).
TASKS = ['task1']
LOCAL = ['--provider', 'local', '--model', MODEL, '--revision', REVISION, '--dtype', 'float16', '--attn', 'sdpa',
         '--tasks', *TASKS]
VARIANTS = {'P0': ['--method', 'zero_shot', '--no-evidence'], 'P1': ['--no-evidence'], 'P2': []}
# Prompt v007 exists for Task 1 only (Task 2 would silently fall back to another prompt).
EXTRA_TRAIN = {'P1-v007-k3': ['--no-evidence', '--prompt', 'v007', '--k', '3'],
               'P2-v007': ['--prompt', 'v007']} if TASKS == ['task1'] else {}
VARIANTS.update(EXTRA_TRAIN)

def run_all(*args):
    return [sys.executable, '-u', str(REPO / 'scripts/run_all.py'), '--train', str(TRAIN), *map(str, args)]

## Smoke: 2 bài train trên GPU 0 (P2, prompt dài nhất)

Xem model trả về JSON đúng chưa và mất bao nhiêu giây một bài. Với Task 1: số giây một bài × khoảng 490 (GPU 0: 160 bài train + 328 public) là thời gian chạy chính.

In [ ]:
smoke = REPO / 'outputs/runs/smoke'
shutil.rmtree(smoke, ignore_errors=True)
subprocess.run(run_all('--data', TRAIN, '--eval', *LOCAL, '--name', 'smoke', '--dry-run'), check=True,
               stdout=subprocess.DEVNULL)
env = dict(os.environ, CUDA_VISIBLE_DEVICES='0', PYTHONUNBUFFERED='1', TOKENIZERS_PARALLELISM='false')
ds = REPO / 'outputs/datasets/sample_dataset'
for task in TASKS:
    start = time.time()
    subprocess.run([sys.executable, '-u', str(REPO / 'scripts/predict.py'), '--config', smoke / f'configs/config_{task}.yaml',
                    '--input', TRAIN, '--output', smoke / f'pred_{task}.json', '--evidence', ds / 'compile.json',
                    '--tests', ds / 'tests.json', '--max-samples', '2'], env=env, check=True)
    print(f'{task}: {time.time() - start:.0f}s for 2 samples (includes ~1-2 min model loading)')
    for line in (smoke / task / 'responses.jsonl').read_text(encoding='utf-8').splitlines():
        row = json.loads(line)
        print('  ', row['sample_id'], '->', str(row.get('raw_response') or row.get('generation_error'))[:300])

## Chạy chính trên 2 GPU song song

- GPU 0: train P0 → P1 → P2 → P1-v007-k3 → P2-v007 (chấm trên 32 bài, để lại một đề thi) → public P1
- GPU 1: public P2

Mỗi GPU có cache riêng: nếu dừng giữa chừng, chạy lại cell này sẽ bỏ qua các bài đã xong.

In [ ]:
import selectors
JOBS = {
    0: [('train', v) for v in VARIANTS] + [('test', 'P1')],
    1: [('test', 'P2')],
}

def commands(gpu):
    cmds = []
    for split, v in JOBS[gpu]:
        data = TRAIN if split == 'train' else TEST
        cmds.append(run_all('--data', data, *(['--eval'] if split == 'train' else []), *LOCAL, *VARIANTS[v],
                            '--cache-path', REPO / f'outputs/qwen4b-cache-gpu{gpu}.jsonl',
                            '--name', f'{split}-qwen35-4b-{v}'))
    return cmds

# One shell chain per GPU; a failed step is reported and the next step still runs.
procs, sel = [], selectors.DefaultSelector()
for gpu in JOBS:
    chain = ' ; '.join(' '.join(f"'{c}'" for c in cmd) + f' || echo "STEP FAILED: {cmd[-1]}"' for cmd in commands(gpu))
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED='1', TOKENIZERS_PARALLELISM='false')
    p = subprocess.Popen(['bash', '-c', chain], env=env, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    procs.append(p)
    sel.register(p.stdout, selectors.EVENT_READ, gpu)
start = time.time()
try:
    while sel.get_map():
        for key, _ in sel.select(timeout=5):
            line = key.fileobj.readline()
            if line:
                print(f'[GPU {key.data} {(time.time() - start) / 60:5.1f}m] {line}', end='', flush=True)
            else:
                sel.unregister(key.fileobj)
finally:
    for p in procs:
        if p.poll() is None:
            p.terminate()
    print('Exit codes:', [p.wait() for p in procs], f'| {(time.time() - start) / 60:.0f} min')

## Kết quả và đóng gói

In [ ]:
runs = REPO / 'outputs/runs'
print('Train (leave-one-exam-out, 32 samples):')
for v in VARIANTS:
    row = {}
    for task, keys in (('task1', ['qwk_total', 'mae_total']), ('task2', ['macro_f1', 'micro_f1'])):
        f = runs / f'train-qwen35-4b-{v}' / f'metrics_{task}.json'
        if f.exists():
            m = json.loads(f.read_text(encoding='utf-8'))['metrics']
            row.update({f'{task}.{k}': round(m[k], 4) for k in keys if k in m})
    print(' ', v, row or 'missing')
for v in ('P1', 'P2'):
    folder = runs / f'test-qwen35-4b-{v}'
    print(v, 'public files:', sorted(p.name for p in folder.glob('submission_*.json')))
    for task in ('task1', 'task2'):
        log = folder / task / 'responses.jsonl'
        if log.exists():
            rows = [json.loads(l) for l in log.read_text(encoding='utf-8').splitlines() if l.strip()]
            print(f'   {task}: {sum(bool(r.get("fallback")) for r in rows)} fallbacks in {len(rows)} log rows')

result = WORK / 'qwen4b_results.zip'
with zipfile.ZipFile(result, 'w', zipfile.ZIP_DEFLATED) as z:
    for folder in runs.glob('*qwen35-4b*'):
        for f in folder.rglob('*'):
            if f.is_file() and 'prepared' not in f.parts:
                z.write(f, f.relative_to(runs))
print('Download', result, round(result.stat().st_size / 2**20, 1), 'MB (private: contains student code in logs)')